In [ ]:
import asyncio
import cv2
import matplotlib.pyplot as plt
import nest_asyncio
import numpy as np
from PIL import Image
import winocr
from ipywidgets import IntSlider, interact

# 1. Patch asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

# 2. Load your image
img_bgr = cv2.imread(r"G:\Programming\Repos\Warframe Overlay\Notebooks\images\image3.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
hls = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HLS)


async def process_and_ocr(lower_hls, upper_hls):
    mask = cv2.inRange(hls, lower_hls, upper_hls)
    inverted_mask = cv2.bitwise_not(mask)

    # Convert to PIL and run Windows OCR
    pil_img = Image.fromarray(inverted_mask)
    results = await winocr.recognize_pil(pil_img, lang="en")

    return mask, results.text


def update_interactive(h_min, h_max, l_min, l_max, s_min, s_max):
    lower = np.array([h_min, l_min, s_min])
    upper = np.array([h_max, l_max, s_max])

    # Safely run the async OCR task via nested loop
    loop = asyncio.get_event_loop()
    mask, detected_text = loop.run_until_complete(
        process_and_ocr(lower, upper)
    )

    # Render image preview
    filtered_result = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(mask, cmap="gray")
    axes[0].set_title("Binary Mask (Inverted for OCR)")
    axes[0].axis("off")

    axes[1].imshow(filtered_result)
    axes[1].set_title("Filtered Color Preview")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    print("================ LIVE WINDOWS OCR OUTPUT ================")
    print(detected_text if detected_text.strip() else "[No text detected]")
    print("========================================================")


# 3. Interactive Sliders
interact(
    update_interactive,
    h_min=IntSlider(min=0, max=179, step=1, value=15, description="Hue Min"),
    h_max=IntSlider(min=0, max=179, step=1, value=35, description="Hue Max"),
    l_min=IntSlider(min=0, max=255, step=1, value=160, description="Light Min"),
    l_max=IntSlider(min=0, max=255, step=1, value=255, description="Light Max"),
    s_min=IntSlider(min=0, max=255, step=1, value=0, description="Sat Min"),
    s_max=IntSlider(min=0, max=255, step=1, value=120, description="Sat Max"),
)

interactive(children=(IntSlider(value=15, description='Hue Min', max=179), IntSlider(value=35, description='Hu…

<function __main__.update_interactive(h_min, h_max, l_min, l_max, s_min, s_max)>